### Gold Layer - What Is It?

In our project, data moves through 3 layers: **Bronze → Silver → Gold**

**Bronze** = Raw data dumped from files. No changes.

**Silver** = Cleaned data. Fixed names, removed duplicates, filtered nulls.

**Gold** = Business-ready tables built for dashboards and reports. This is where analysts actually query data.

---

### What's different about Gold?

In Silver, each table is still **one entity** (circuits, races, drivers, etc.) — just cleaned up.

In Gold, we **combine multiple silver tables** into fewer, wider tables that answer business questions directly. No more joining at query time.

| Silver (separate tables) | Gold (combined tables) |
| --- | --- |
| `races` + `circuits` (2 tables) | `dim_races` (1 table with all info) |
| `drivers` + region lookup | `dim_drivers` (drivers with their region) |
| `constructors` + region lookup | `dim_constructors` (teams with their region) |
| `results` + `sprints` (2 tables) | `facts_session_results` (one unified table) |

---

### Types of Gold Tables

We build two types:

**Dimension tables** (prefix: `dim_`) = WHO/WHAT/WHERE
- `dim_races` → Which race, which circuit, which country
- `dim_drivers` → Which driver, their nationality, their region
- `dim_constructors` → Which team, their nationality, their region

**Fact tables** (prefix: `facts_`) = WHAT HAPPENED
- `facts_session_results` → Who finished where, how many points, did they win

**Reference tables** (prefix: `ref_`) = LOOKUP DATA
- `ref_nationalaty_regions` → Maps nationality to region (e.g., "British" → "Europe")

---

### How each Gold notebook works

Every gold notebook follows this pattern:
1. Load config + gold helpers (variables and `write_to_gold()` function)
2. Set the target table name
3. Read silver tables (filtered by batch where applicable)
4. Join/transform/derive new columns
5. Write to gold using `write_to_gold()`

---

### Utilities used (from `00-common` folder):

| File | What it does |
| --- | --- |
| `01.environment-config` | Stores shared variables: `catalog_name` = `formula1_incr`, `silver_schema` = `silver`, `gold_schema` = `gold` |
| `04.gold_helpers` | Contains the `write_to_gold()` function that all gold notebooks call to save data |

---

### What does `write_to_gold()` do?

Same idea as `write_to_silver()` but for gold tables:

1. Adds `created_timestamp` and `updated_timestamp` columns
2. Checks if the table exists
3. **If NO** → creates the table from scratch (first run)
4. **If YES** → merges new data in:
   - Rows that match the merge condition → **updates** the listed columns
   - Rows that don't match → **inserts** them as new

The only difference from `write_to_silver()`: gold doesn't check `batch_id >= t.batch_id` condition on update — it always updates matched rows.

---

### Notebook 01: Build Races Dimension
**Target:** `formula1_incr.gold.dim_races`

**What it does:**
- Reads `silver.circuits` and `silver.races`
- Joins them on `circuit_id` (inner join)
- Result: one row per race with race name, date, circuit name, city, country

**Why?** So dashboards can show race info without needing to join circuits every time.

**Merge key:** `season` + `round` (each race is unique by season and round number)

---

### Notebook 02: Build Constructors Dimension
**Target:** `formula1_incr.gold.dim_constructors`

**What it does:**
- Reads `silver.constructors` and `gold.ref_nationalaty_regions`
- Left outer join on `nationality` to add a `nationality_region` column
- Result: one row per team with name, nationality, and region

**Why?** So we can group/filter teams by region (Europe, Asia, etc.) in reports.

**Merge key:** `constructor_id`

---

### Notebook 03: Build Drivers Dimension
**Target:** `formula1_incr.gold.dim_drivers`

**What it does:**
- Reads `silver.drivers` and `gold.ref_nationalaty_regions`
- Left outer join on `nationality` to add a `nationality_region` column
- Result: one row per driver with name, DOB, nationality, and region

**Why?** So we can group/filter drivers by region in reports.

**Merge key:** `driver_id`

---

### Notebook 04: Build Session Results Fact
**Target:** `formula1_incr.gold.facts_session_results`

**What it does:**
- Reads `silver.results` → adds `session_type = 'RACE'`
- Reads `silver.sprints` → adds `session_type = 'SPRINT'`
- Unions both into one table
- Derives 3 useful boolean columns:

| Column | Meaning | Example use |
| --- | --- | --- |
| `is_win` | Did the driver finish 1st? | `SUM(is_win)` = total wins |
| `is_podium` | Did the driver finish top 3? | `WHERE is_podium = true` |
| `has_points` | Did the driver score any points? | Count points finishes |

**Why?** One table for ALL session results (race + sprint). Boolean columns make dashboard queries simpler.

**Merge key:** `season` + `round` + `driver_id` + `constructor_id` + `session_type`

---

### Notebook 11: Nationality Reference Table
**Target:** `formula1_incr.gold.ref_nationalaty_regions`

**What it does:**
- Manually defines a mapping of 41 nationalities to 6 regions
- Example: "British" → "Europe", "Japanese" → "Asia", "Brazilian" → "South America"
- Used by both `dim_drivers` and `dim_constructors` during their join step

**Why?** Without this, we'd have no way to group by region.

---

### How Gold tables connect (Star Schema)

The fact table sits in the center. Dimension tables surround it:

```
                    dim_races
                  (season, round)
                        |
                        |
  dim_constructors ---- facts_session_results ---- dim_drivers
  (constructor_id)      (the central fact)         (driver_id)
```

To answer "How many wins did British drivers score at European circuits in 2025?":
- Join `facts_session_results` with `dim_drivers` (filter region = Europe)
- Join with `dim_races` (filter season = 2025)
- Filter `is_win = true`
- Count rows

---

### Summary of Gold Tables

| Table | Type | Key Columns | Source |
| --- | --- | --- | --- |
| `ref_nationalaty_regions` | Reference | nationality, region | Manual |
| `dim_races` | Dimension | season, round, race_name, circuit_name, country | silver.races + silver.circuits |
| `dim_constructors` | Dimension | constructor_id, constructor_name, nationality_region | silver.constructors + gold.ref |
| `dim_drivers` | Dimension | driver_id, driver_name, nationality_region | silver.drivers + gold.ref |
| `facts_session_results` | Fact | season, round, driver_id, constructor_id, points, is_win | silver.results + silver.sprints |

In [0]:
displayHTML('<iframe style="border: 1px solid rgba(0, 0, 0, 0.1);" width="800" height="450" src="https://embed.figma.com/board/sqzbUFRL8caRHqFrJgYmxi/Formula1-Gold-Schema-ERD---Matching-Colors?node-id=0-1&embed-host=share" allowfullscreen></iframe>')